# KuchoLM NIDA2 — 7M pointer/copy curriculum

NIDA2 は「知らないものを無理に生成せず、入力からコピーする」ことを重視した約7Mの口調変換モデルです。

- 12k BPE + byte fallback
- COPY warmup 3 epochs
- mixed 最大12 epochs
- embedding/position std=0.02
- pointer-generator style copy distribution
- generation / copy gate
- quality scoreでBEST選択
- style終端で即停止
- early stopping
- `ねこ`・ID・未知Unicode・固有名詞の保持を固定テスト


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets sentencepiece torch


In [ ]:
from pathlib import Path
import difflib, json, math, random, re, string
import MeCab, sentencepiece as spm, torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import Dataset, DataLoader

DATA_PATH=Path('/content/kucholm_nida.jsonl')
WORK_DIR=Path('/content/kucholm_work'); WORK_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS=50_000
VOCAB_SIZE=12_000
MAX_LEN=160
COPY_RATIO=0.60
REF_RATIO=0.40
COPY_WARMUP_ROWS=30_000
COPY_WARMUP_EPOCHS=3
MIXED_EPOCHS=12
EOS_WEIGHT=1.5
BASE_LR=3e-4
MIN_LR=8e-5
QUALITY_SAMPLES=48
EARLY_STOP_PATIENCE=3
MIN_MIXED_EPOCHS=3
SEED=42

random.seed(SEED)
torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tagger=MeCab.Tagger()
print('device:',device)


In [ ]:
URL_RE=re.compile(r'https?://|www\.|```|`[^`]+`')
SENTENCE_RE=re.compile(r'(.+?[。！？!?]+|.+$)',re.S)

def parse_tokens(text):
    node=tagger.parseToNode(text); out=[]
    while node:
        if node.surface:
            f=node.feature.split(',')
            out.append({
                'surface':node.surface,
                'pos':f[0] if len(f)>0 else '',
                'ctype':f[4] if len(f)>4 else '*',
                'lemma':f[7] if len(f)>7 else '*',
                'orth_base':f[10] if len(f)>10 else '*',
            })
        node=node.next
    return out

def dictionary_form(t):
    for k in ('orth_base','lemma'):
        v=t.get(k,'*')
        if v not in ('','*') and re.search(r'[ぁ-ん一-龯]',v):
            return v
    return t['surface']

def is_ichidan(t):
    c=t.get('ctype','')
    return '下一段' in c or '上一段' in c or '一段' in c

def ta_form(base,t):
    if base=='行く': return '行った'
    if base=='来る': return '来た'
    if base=='する': return 'した'
    if is_ichidan(t): return base[:-1]+'た'
    if base.endswith(('う','つ','る')): return base[:-1]+'った'
    if base.endswith(('む','ぶ','ぬ')): return base[:-1]+'んだ'
    if base.endswith('く'): return base[:-1]+'いた'
    if base.endswith('ぐ'): return base[:-1]+'いだ'
    if base.endswith('す'): return base[:-1]+'した'
    return base+'た'

def nai_form(base,t):
    if base=='する': return 'しない'
    if base=='来る': return '来ない'
    if is_ichidan(t): return base[:-1]+'ない'
    if base.endswith('う'): return base[:-1]+'わない'
    table={'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    return base[:-1]+table[base[-1]]+'ない' if base[-1:] in table else base+'ない'

def convert_polite_tail(body):
    rules=[
      (r'ということでした$','ってことだった'),(r'ということです$','ってこと'),
      (r'かもしれません$','かもしれない'),(r'わかりません$','わからない'),
      (r'知りません$','知らない'),(r'いけません$','いけない'),
      (r'ありません$','ない'),(r'ございました$','あった'),(r'ございます$','ある'),
      (r'てきました$','てきた'),(r'て来ました$','て来た'),
      (r'てしまいました$','てしまった'),(r'でしまいました$','でしまった'),
      (r'でした$','だった'),(r'です$','')]
    for p,r in rules:
        if re.search(p,body): return re.sub(p,r,body)

    toks=parse_tokens(body); sur=[t['surface'] for t in toks]
    for suffix,mode in [
        (['ませ','ん','でし','た'],'negative_past'),
        (['ませ','ん'],'negative'),
        (['まし','た'],'past'),
        (['ます'],'present'),
    ]:
        if len(sur)<len(suffix) or sur[-len(suffix):]!=suffix:
            continue
        end=len(toks)-len(suffix)
        vi=next((i for i in range(end-1,-1,-1) if toks[i]['pos']=='動詞'),None)
        if vi is None:
            continue
        tok=toks[vi]; base=dictionary_form(tok)
        prefix=''.join(x['surface'] for x in toks[:vi])
        if mode=='present': repl=base
        elif mode=='past': repl=ta_form(base,tok)
        else:
            neg=nai_form(base,tok)
            repl=neg if mode=='negative' else neg[:-2]+'なかった'
        return prefix+repl
    return body

def convert_sentence(sentence):
    m=re.match(r'^(\s*)(.*?)(\s*)$',sentence,re.S)
    leading,core,trailing=m.groups()
    if not core or URL_RE.search(core):
        return sentence
    pm=re.search(r'([。！？!?]+)$',core)
    punct=pm.group(1) if pm else ''
    body=core[:-len(punct)] if punct else core
    converted=convert_polite_tail(body)
    is_q=bool(re.search(r'[？?]$',punct))
    if converted.endswith(('ね','よ','な')):
        converted=converted[:-1]+'ニダ'+converted[-1]
    else:
        converted += 'ニカ' if is_q else 'ニダよ'
    return leading+converted+punct+trailing

def to_nida(text):
    if not text or URL_RE.search(text):
        return None
    return ''.join(convert_sentence(m.group(0)) for m in SENTENCE_RE.finditer(text))

if not DATA_PATH.exists():
    ds=load_dataset('range3/cc100-ja',split='train',streaming=True)
    n=0
    with DATA_PATH.open('w',encoding='utf-8') as f:
        for row in ds:
            s=str(row['text'])
            if len(s.strip())<2 or len(s)>220:
                continue
            t=to_nida(s)
            if not t or t==s:
                continue
            f.write(json.dumps({'source':s,'target':t},ensure_ascii=False)+'\n')
            n+=1
            if n>=MAX_ROWS:
                break
    print('written:',n)
else:
    print('using existing:',DATA_PATH)


In [ ]:
raw=[]
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        x=json.loads(line)
        raw.append((x['source'],x['target']))

random.shuffle(raw)
raw=raw[:MAX_ROWS]
cut=max(1,int(len(raw)*0.98))
train_raw,val_raw=raw[:cut],raw[cut:]

RARE_CHARS='髙﨑𠮷神邉邊齋齊塚𩸽'
ASCII_POOL=string.ascii_uppercase+string.digits

def make_ref():
    a=''.join(random.choices(ASCII_POOL,k=7))
    b=''.join(random.choices(ASCII_POOL,k=5))
    return f'[REF:{a}-{b}/{random.choice(RARE_CHARS)}{random.choice(RARE_CHARS)}]'

nida_rows=[]
copy_rows=[]
for s,t in train_raw:
    nida_rows.append((f'<NIDA_FICTION> {s}',t))
    copy_rows.append((f'<COPY> {s}',s))
    if random.random()<REF_RATIO:
        ref=make_ref()
        nida_rows.append((f'<NIDA_FICTION> {ref} {s}',f'{ref} {t}'))
        copy_rows.append((f'<COPY> {ref} {s}',f'{ref} {s}'))

mixed=nida_rows+random.sample(
    copy_rows,
    min(len(copy_rows),int(len(nida_rows)*COPY_RATIO))
)
random.shuffle(mixed)

spm_input=WORK_DIR/'spm_train.txt'
with spm_input.open('w',encoding='utf-8') as f:
    for s,t in mixed:
        f.write(s.replace('\n',' ')+'\n')
        f.write(t.replace('\n',' ')+'\n')

spm.SentencePieceTrainer.train(
    input=str(spm_input),
    model_prefix=str(WORK_DIR/'kucholm_spm'),
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=1.0,
    byte_fallback=True,
    normalization_rule_name='identity',
    split_digits=True,
    hard_vocab_limit=False,
    pad_id=0,unk_id=1,bos_id=2,eos_id=3,
    user_defined_symbols=['<NIDA_FICTION>','<COPY>']
)

sp=spm.SentencePieceProcessor(model_file=str(WORK_DIR/'kucholm_spm.model'))
PAD,UNK,BOS,EOS=0,1,2,3
VOCAB=sp.vocab_size()
NIDA_ID=sp.piece_to_id('<NIDA_FICTION>')
COPY_ID=sp.piece_to_id('<COPY>')
SPECIAL_COPY_BLOCK={PAD,BOS,EOS,NIDA_ID,COPY_ID}

print('vocab:',VOCAB,'NIDA_ID:',NIDA_ID,'COPY_ID:',COPY_ID)
for x in ['ねこ','髙﨑𠮷野家ABC-123','𩸽を食べました。','KuchoLM-X7-2026']:
    ids=sp.encode(x,out_type=int)
    print(x,'->',sp.decode(ids),'unk:',ids.count(UNK),'tokens:',len(ids))


In [ ]:
def encode(text):
    return [BOS]+sp.encode(text,out_type=int)+[EOS]

def fits(pair):
    return len(encode(pair[0]))<=MAX_LEN and len(encode(pair[1]))<=MAX_LEN

copy_rows=[p for p in copy_rows if fits(p)]
mixed=[p for p in mixed if fits(p)]
val_rows=[(f'<NIDA_FICTION> {s}',t) for s,t in val_raw]
val_rows=[p for p in val_rows if fits(p)]

class PairDataset(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        s,t=self.rows[i]
        return torch.tensor(encode(s)),torch.tensor(encode(t))

def collate(batch):
    s,t=zip(*batch)
    return (
        nn.utils.rnn.pad_sequence(s,batch_first=True,padding_value=PAD),
        nn.utils.rnn.pad_sequence(t,batch_first=True,padding_value=PAD),
    )

BATCH=64 if device.type=='cuda' else 8
args={
    'batch_size':BATCH,
    'collate_fn':collate,
    'pin_memory':device.type=='cuda',
    'num_workers':2 if device.type=='cuda' else 0,
}
copy_loader=DataLoader(PairDataset(copy_rows[:COPY_WARMUP_ROWS]),shuffle=True,**args)
mixed_loader=DataLoader(PairDataset(mixed),shuffle=True,**args)
val_loader=DataLoader(PairDataset(val_rows),shuffle=False,**args)


In [ ]:
D_MODEL=224
NHEAD=8
ENC_LAYERS=3
DEC_LAYERS=3
FF=896

class KuchoNIDA2(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(VOCAB,D_MODEL,padding_idx=PAD)
        self.pos=nn.Embedding(MAX_LEN,D_MODEL)
        self.tf=nn.Transformer(
            d_model=D_MODEL,
            nhead=NHEAD,
            num_encoder_layers=ENC_LAYERS,
            num_decoder_layers=DEC_LAYERS,
            dim_feedforward=FF,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.lm_head=nn.Linear(D_MODEL,VOCAB,bias=False)

        # 1 = generate from vocabulary, 0 = copy from source.
        self.copy_gate=nn.Linear(D_MODEL*2,1)

        nn.init.normal_(self.embed.weight,0.0,0.02)
        nn.init.normal_(self.pos.weight,0.0,0.02)
        nn.init.zeros_(self.copy_gate.weight)
        nn.init.constant_(self.copy_gate.bias,-0.7)  # start slightly copy-biased
        with torch.no_grad():
            self.embed.weight[PAD].zero_()
        self.lm_head.weight=self.embed.weight

    def add_pos(self,ids):
        p=torch.arange(ids.size(1),device=ids.device).unsqueeze(0)
        return self.embed(ids)+self.pos(p)

    def forward(self,source,target,return_gate=False):
        src_pad=source.eq(PAD)
        tgt_pad=target.eq(PAD)
        tgt_len=target.size(1)
        causal=torch.triu(
            torch.ones(tgt_len,tgt_len,dtype=torch.bool,device=target.device),
            diagonal=1,
        )

        src_e=self.add_pos(source)
        memory=self.tf.encoder(src_e,src_key_padding_mask=src_pad)
        tgt_h=self.tf.decoder(
            self.add_pos(target),
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        )

        gen_probs=torch.softmax(self.lm_head(tgt_h),dim=-1)

        # Pointer distribution over source positions.
        copy_scores=torch.matmul(tgt_h,memory.transpose(1,2))/math.sqrt(D_MODEL)
        copy_mask=src_pad.clone()
        for special_id in SPECIAL_COPY_BLOCK:
            copy_mask |= source.eq(special_id)
        copy_scores=copy_scores.masked_fill(copy_mask.unsqueeze(1),-1e4)
        copy_attn=torch.softmax(copy_scores,dim=-1)

        # Convert source-position probabilities to vocabulary-token probabilities.
        copy_vocab=torch.zeros(
            source.size(0),target.size(1),VOCAB,
            dtype=gen_probs.dtype,device=source.device
        )
        src_ids=source.unsqueeze(1).expand(-1,target.size(1),-1)
        copy_vocab.scatter_add_(2,src_ids,copy_attn)

        context=torch.matmul(copy_attn,memory)
        p_gen=torch.sigmoid(self.copy_gate(torch.cat([tgt_h,context],dim=-1)))
        probs=p_gen*gen_probs+(1.0-p_gen)*copy_vocab
        probs=probs.clamp_min(1e-9)

        if return_gate:
            return probs,p_gen.squeeze(-1),copy_attn
        return probs

model=KuchoNIDA2().to(device)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.3f}M parameters')
print('rows:',len(copy_rows[:COPY_WARMUP_ROWS]),len(mixed),len(val_rows),'batch:',BATCH)


In [ ]:
TESTS=[
    '今日は学校です。',
    'ねこが好きです。',
    '明日は雨が降るかもしれません。',
    '最近少し暖かくなってきました。',
    '製品KuchoLM-X7-2026は正常に動作しています。',
    '髙﨑𠮷野家ABC-123を確認しました。',
    '𩸽を食べました。',
]
STYLE_ENDINGS=('ニダよ。','ニダよ！','ニダよ!','ニカ？','ニカ?','ニダね。','ニダな。')

@torch.no_grad()
def infer(text,tag='<NIDA_FICTION>',show_gate=False):
    source_ids=encode(f'{tag} {text}')
    source=torch.tensor([source_ids],device=device)
    output=[BOS]
    limit=min(MAX_LEN-1,len(source_ids)+12)
    gate_trace=[]

    for _ in range(limit):
        probs,p_gen,_=model(
            source,
            torch.tensor([output],device=device),
            return_gate=True,
        )
        scores=probs[0,-1].clone()
        gate_trace.append(float(p_gen[0,-1]))

        if len(output)>=3:
            prefix=tuple(output[-2:])
            banned={
                output[i+2]
                for i in range(len(output)-2)
                if tuple(output[i:i+2])==prefix
            }
            if banned:
                scores[list(banned)]=0.0

        if len(output)>=2 and output[-1]==output[-2]:
            scores[output[-1]]=0.0

        nxt=int(torch.argmax(scores))
        if nxt==EOS:
            break
        output.append(nxt)

        partial=sp.decode(output[1:])
        if tag=='<NIDA_FICTION>' and partial.endswith(STYLE_ENDINGS):
            break
        if tag=='<COPY>' and partial==text:
            break

    text_out=sp.decode(output[1:])
    if show_gate:
        mean_gate=sum(gate_trace)/max(1,len(gate_trace))
        return text_out,mean_gate
    return text_out

def show_samples(label,tag='<NIDA_FICTION>'):
    model.eval()
    print('\n---',label,'---')
    for s in TESTS:
        out,g=infer(s,tag,show_gate=True)
        print(s,'->',out,f'[p_gen={g:.3f}, copy={1-g:.3f}]')


In [ ]:
optimizer=torch.optim.AdamW(
    model.parameters(),lr=BASE_LR,betas=(0.9,0.98),weight_decay=0.01
)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,T_max=MIXED_EPOCHS,eta_min=MIN_LR
)
scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')

def loss_fn(probs,target):
    y=target.reshape(-1)
    flat=probs.reshape(-1,VOCAB)
    valid=y!=PAD
    idx=torch.arange(y.numel(),device=y.device)
    token_probs=flat[idx,y].clamp_min(1e-9)
    losses=-torch.log(token_probs)
    weights=torch.ones_like(losses)
    weights[y==EOS]=EOS_WEIGHT
    return (losses[valid]*weights[valid]).sum()/weights[valid].sum()

def train_one(loader):
    model.train()
    total=0.0
    for s,t in loader:
        s=s.to(device,non_blocking=True)
        t=t.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=device.type=='cuda'):
            probs=model(s,t[:,:-1])
            loss=loss_fn(probs,t[:,1:])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()
        total+=loss.item()
    return total/max(1,len(loader))

@torch.no_grad()
def validation_loss():
    model.eval()
    total=0.0
    for s,t in val_loader:
        s=s.to(device,non_blocking=True)
        t=t.to(device,non_blocking=True)
        total+=loss_fn(model(s,t[:,:-1]),t[:,1:]).item()
    return total/max(1,len(val_loader))

def repetition_rate(text):
    p=sp.encode(text,out_type=str)
    if len(p)<2:
        return 0.0
    return sum(p[i]==p[i-1] for i in range(1,len(p)))/(len(p)-1)

def protected_chunks(text):
    # ASCII identifiers + non-basic Unicode characters that should survive unchanged.
    ascii_chunks=re.findall(r'[A-Za-z0-9][A-Za-z0-9._:/+\-]{1,}',text)
    rare=[c for c in text if ord(c)>0xFFFF or c in RARE_CHARS]
    return list(dict.fromkeys(ascii_chunks+rare))

@torch.no_grad()
def quality_metrics(limit=QUALITY_SAMPLES):
    model.eval()
    sims=[]; reps=[]; lengths=[]; protected=[]; gates=[]
    for tagged,expected in val_rows[:limit]:
        src=tagged.removeprefix('<NIDA_FICTION> ')
        actual,g=infer(src,show_gate=True)
        sims.append(difflib.SequenceMatcher(None,expected,actual).ratio())
        reps.append(repetition_rate(actual))
        lengths.append(abs(len(actual)-len(expected))/max(1,len(expected)))
        chunks=protected_chunks(src)
        protected.append(
            1.0 if not chunks else sum(c in actual for c in chunks)/len(chunks)
)
        gates.append(g)

    sim=sum(sims)/max(1,len(sims))
    rep=sum(reps)/max(1,len(reps))
    length_pen=sum(min(x,1.0) for x in lengths)/max(1,len(lengths))
    keep=sum(protected)/max(1,len(protected))
    pgen=sum(gates)/max(1,len(gates))
    score=sim+0.22*keep-0.30*rep-0.18*length_pen
    return {
        'score':score,
        'similarity':sim,
        'protected':keep,
        'repetition':rep,
        'length_penalty':length_pen,
        'p_gen':pgen,
        'copy_gate':1-pgen,
    }

for w in range(1,COPY_WARMUP_EPOCHS+1):
    loss=train_one(copy_loader)
    torch.save(
        {'model':model.state_dict(),'loss':loss},
        WORK_DIR/f'KuchoLM-NIDA2-COPY-warmup{w}.pt'
)
    print(f'COPY warmup {w}: {loss:.4f}')
    show_samples(f'COPY warmup {w}',tag='<COPY>')

best_score=-1e9
best_path=WORK_DIR/'KuchoLM-NIDA2-7M.pt'
bad_epochs=0

for epoch in range(1,MIXED_EPOCHS+1):
    train_loss=train_one(mixed_loader)
    val_loss=validation_loss()
    q=quality_metrics()
    lr=optimizer.param_groups[0]['lr']

    ckpt={
        'model':model.state_dict(),
        'val':val_loss,
        'quality':q,
        'epoch':epoch,
        'lr':lr,
        'model_name':'KuchoLM-NIDA2-7M',
    }
    torch.save(ckpt,WORK_DIR/f'KuchoLM-NIDA2-7M-epoch{epoch}.pt')

    print(
        f"epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f} "
        f"quality={q['score']:.4f} sim={q['similarity']:.3f} "
        f"keep={q['protected']:.3f} rep={q['repetition']:.3f} "
        f"lenpen={q['length_penalty']:.3f} copy={q['copy_gate']:.3f} lr={lr:.2e}"
)
    show_samples(f'epoch {epoch}')

    if q['score']>best_score+0.002:
        best_score=q['score']
        bad_epochs=0
        torch.save(ckpt,best_path)
    else:
        bad_epochs+=1

    scheduler.step()
    if epoch>=MIN_MIXED_EPOCHS and bad_epochs>=EARLY_STOP_PATIENCE:
        print('early stopping: generation quality did not improve')
        break

best=torch.load(best_path,map_location=device)
model.load_state_dict(best['model'])
print('BEST epoch:',best.get('epoch'),'quality:',best.get('quality'))
show_samples('BEST')


In [ ]:
q=quality_metrics(limit=min(100,len(val_rows)))
copy_eval=val_raw[:100]
copy_acc=sum(infer(s,tag='<COPY>')==s for s,_ in copy_eval)/max(1,len(copy_eval))

stress=[
    'ねこが好きです。',
    'ねこねこねこです。',
    '𩸽を食べました。',
    '髙﨑𠮷野家ABC-123を確認しました。',
    '新製品QZX-9999-β版を使っています。',
    '🐈とねこが好きです。',
]
print('\n--- NIDA2 stress test ---')
for s in stress:
    out,g=infer(s,show_gate=True)
    print(s,'->',out,f'[copy={1-g:.3f}]')

print('\nquality:',q)
print('COPY exact accuracy:',copy_acc)
print('model:',WORK_DIR/'KuchoLM-NIDA2-7M.pt')
print('tokenizer:',WORK_DIR/'kucholm_spm.model')
